In [1]:
import pandas as pd

In [24]:
from data_formatter import collect_an_format_data, format_data
from Operations import *

In [25]:

pc = PercentChange(['bb_upper'])

In [26]:
df = collect_an_format_data(
    "test_0",
    [ RemoveColumns(['time','atr',"bb_upper","bb_middle","bb_lower"]), pc],
    column_merge_mode='merge',
    write_csv=True,
)

Skipping.. RemoveColumns for ANET2025-08-12.csv
Skipping.. PercentChange for ANET2025-08-12.csv
Skipping.. PercentChange for RVNL.NS2025-08-12.csv
Skipping.. PercentChange for strange.csv


In [27]:
# What if some cols are thern and some arent then what

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1530 entries, 0 to 1529
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   action          1530 non-null   object 
 1   close           1530 non-null   float64
 2   open            1529 non-null   float64
 3   high            1530 non-null   float64
 4   low             1530 non-null   float64
 5   volume          1530 non-null   int64  
 6   price           1530 non-null   float64
 7   fast_ema        1530 non-null   float64
 8   slow_ema        1530 non-null   float64
 9   rsi_14          1530 non-null   float64
 10  vwma            1530 non-null   float64
 11  action_quality  1530 non-null   object 
dtypes: float64(9), int64(1), object(2)
memory usage: 143.6+ KB


In [29]:
_df = df

In [30]:
df = format_data(df,operations=[RemoveEmptyNullRows()])

In [31]:
df

,action,close,open,high,low,volume,price,fast_ema,slow_ema,rsi_14,vwma,action_quality
0,hold,138.500000,138.544998,139.085098,138.279999,212860.0,138.500000,0.000000,0.000000,50.000000,138.500000,good
1,hold,138.270004,138.639999,138.800003,138.199997,28967.0,138.270004,0.000000,0.000000,50.000000,138.472450,neutral
2,hold,138.990005,138.270004,139.000000,138.254593,33886.0,138.990005,0.000000,0.000000,50.000000,138.536059,good
3,hold,138.910004,139.074997,139.110001,138.789993,21643.0,138.910004,0.000000,0.000000,50.000000,138.563277,neutral
4,hold,139.695007,138.970001,139.725006,138.910004,36273.0,139.695007,0.000000,0.000000,50.000000,138.686321,good
...,...,...,...,...,...,...,...,...,...,...,...,...
1525,hold,326.250000,326.149994,326.299988,325.600006,77197.0,326.250000,327.688035,331.070103,25.721164,329.585170,neutral
1526,hold,326.250000,326.250000,326.850006,326.000000,78862.0,326.250000,327.400428,330.631912,25.721164,329.446522,neutral
1527,hold,323.700012,326.500000,326.649994,323.700012,195146.0,323.700012,326.660345,330.001739,21.132866,329.004236,neutral
1528,hold,322.250000,323.649994,324.000000,322.000000,157694.0,322.250000,325.778276,329.297035,19.051673,328.616187,neutral


In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1530 entries, 0 to 1529
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   action          1529 non-null   object 
 1   close           1529 non-null   float64
 2   open            1529 non-null   float64
 3   high            1529 non-null   float64
 4   low             1529 non-null   float64
 5   volume          1529 non-null   float64
 6   price           1529 non-null   float64
 7   fast_ema        1529 non-null   float64
 8   slow_ema        1529 non-null   float64
 9   rsi_14          1529 non-null   float64
 10  vwma            1529 non-null   float64
 11  action_quality  1529 non-null   object 
dtypes: float64(10), object(2)
memory usage: 143.6+ KB


In [33]:
_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1530 entries, 0 to 1529
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   action          1529 non-null   object 
 1   close           1529 non-null   float64
 2   open            1529 non-null   float64
 3   high            1529 non-null   float64
 4   low             1529 non-null   float64
 5   volume          1529 non-null   float64
 6   price           1529 non-null   float64
 7   fast_ema        1529 non-null   float64
 8   slow_ema        1529 non-null   float64
 9   rsi_14          1529 non-null   float64
 10  vwma            1529 non-null   float64
 11  action_quality  1529 non-null   object 
dtypes: float64(10), object(2)
memory usage: 143.6+ KB


In [ ]:
def process_df( df, column_generator:list=[] ):
    for col_name, columns, func in column_generator:
        df[col_name] = func( df[columns]  ) # columns is a list of string
   
   
    # Price returns and differences
    df['return_open_close'] = (df['close'] - df['open']) / df['open']
    df['return_prev_close'] = df['close'].pct_change()
    df['rolling_return_3'] = df['return_prev_close'].rolling(window=3).mean()
    df['rolling_return_5'] = df['return_prev_close'].rolling(window=5).mean()

    # Volatility/ATR
    high, low, close = df['high'], df['low'], df['close']
    prev_close = close.shift(1)
    tr = pd.concat([
        (high - low),
        (high - prev_close).abs(),
        (low - prev_close).abs()
    ], axis=1).max(axis=1)

    # Volumne Features
    df['volume_pct_change'] = df['volume'].pct_change()
    df['volume_rolling_mean_5'] = df['volume'].rolling(window=5).mean()
    df['volume_rolling_mean_10'] = df['volume'].rolling(window=10).mean()
    
    
    df['atr_14'] = tr.rolling(window=14, min_periods=1).mean()
    df['rolling_std_5'] = df['return_prev_close'].rolling(window=5).std()
    df['rolling_std_10'] = df['return_prev_close'].rolling(window=10).std()


    


    df = df.reset_index(drop=True)
    return df

In [ ]:
import glob

all_files = glob.glob('simulation_results/*.csv')
dfs = []

for file in all_files:
    df = pd.read_csv(file)
    # Optionally add identifiers if needed
    # df['date'] = extract_date_from_filename(file)
    # df['symbol'] = extract_symbol_from_filename(file)
    df = process_df(df)
    dfs.append(df)

full_data = pd.concat(dfs, ignore_index=True)


In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Assuming full_data has all feature-engineered columns and 'action_quality' and 'time'

# 1. Encode target variable
le = LabelEncoder()
full_data['action_quality_enc'] = le.fit_transform(full_data['action_quality'])

# 2. Sort data by time (ascending)
full_data = full_data.sort_values('time').reset_index(drop=True)

# 3. Separate features and target
X = full_data.drop(columns=['action_quality', 'action_quality_enc', 'time'])
y = full_data['action_quality_enc']

# 4. Split into train and test sets (80/20 by time)
split_idx = int(len(full_data) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# 5. Compute class weights for imbalance handling
classes = list(le.transform(le.classes_))
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))

# 6. (Optional) Feature scaling for models that need it (skip if using tree-based models)
# scaler = StandardScaler()
# X_train = scaler.fit_transform(X_train)
# X_test = scaler.transform(X_test)

# 7. Train Random Forest classifier with class weights
clf = RandomForestClassifier(class_weight=class_weight_dict, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

# 8. Make predictions and evaluate
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=le.classes_))


In [ ]:
# scaler is already fitted from training phase

X_new_scaled = scaler.transform([X_new])  # Scale new data point using training params
prediction = model.predict(X_new_scaled)


In [ ]:
import joblib

# After training
joblib.dump(model, 'decider_model.pkl')
joblib.dump(scaler, 'scaler.pkl')


In [ ]:
model = joblib.load('decider_model.pkl')
scaler = joblib.load('scaler.pkl')

# Prepare new data features (X_new) same way as training
X_new_scaled = scaler.transform([X_new])
prediction = model.predict(X_new_scaled)
